# Stage 1 — PDF Ingestion, Chunking & Delta Storage

First stage of the RAG Technical Knowledge Assistant. Reads the two source PDFs from the Unity Catalog volume `dev_project.rag_docs.source_pdfs`, extracts text page by page (pypdf), cleans it, chunks it (~500 tokens with ~100-token overlap), and stores all chunks idempotently in the Delta table `dev_project.rag_docs.document_chunks`.

**Scope: Stage 1 only — no embeddings, no Vector Search, no LLM calls.**

In [0]:
%pip install -q pypdf tiktoken

dbutils.library.restartPython()

In [0]:
import os
import shutil
import sys

# --- The six pipeline modules live in the same folder as this notebook ---
NOTEBOOK_DIR = os.getcwd()
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

# --- One-time infrastructure: schema + volume inside dev_project, PDFs copied in ---
spark.sql("CREATE SCHEMA IF NOT EXISTS dev_project.rag_docs")
spark.sql("CREATE VOLUME IF NOT EXISTS dev_project.rag_docs.source_pdfs")

SOURCE_PDF_DIR = "/Workspace/Users/dsw4010@gmail.com/First-Repo_Databricks_practice/Rag_Folder/pdfs"
VOLUME_DIR = "/Volumes/dev_project/rag_docs/source_pdfs"
os.makedirs(VOLUME_DIR, exist_ok=True)

for f in sorted(os.listdir(SOURCE_PDF_DIR)):
    if f.lower().endswith(".pdf"):
        dst = os.path.join(VOLUME_DIR, f)
        if os.path.exists(dst):
            print(f"already in volume: {f}")
        else:
            shutil.copy2(os.path.join(SOURCE_PDF_DIR, f), dst)
            print(f"copied to volume:  {f}")

print("\nVolume contents:", sorted(os.listdir(VOLUME_DIR)))

# --- Pipeline configuration ---
from config import IngestionConfig
from ingestion_pipeline import run_ingestion

cfg = IngestionConfig()
print(f"\nsource volume : {cfg.volume_path}")
print(f"target table  : {cfg.full_table_name}")
print(f"chunking      : ~{cfg.chunk_size_tokens} tokens per chunk, ~{cfg.chunk_overlap_tokens} token overlap")

In [0]:
# First ingestion run: load -> parse -> clean -> chunk -> MERGE into Delta
summary = run_ingestion(cfg, spark)

import pandas as pd

display(pd.DataFrame(summary["documents"]))
print(f"documents processed: {summary['documents_processed']} | chunks prepared: {summary['chunks_prepared']}")

In [0]:
# Display extracted text: parse each PDF again and show cleaned page text
from document_parser import parse_pdf
from text_cleaner import clean_text

for f in sorted(os.listdir(VOLUME_DIR)):
    if not f.lower().endswith(".pdf"):
        continue
    pages = parse_pdf(os.path.join(VOLUME_DIR, f), f)
    print("=" * 100)
    print(f"{f} — {len(pages)} pages yielded text")
    print("=" * 100)
    for p in pages[:2]:
        print(f"\n--- page {p.page_number} (cleaned, first 600 chars) ---")
        print(clean_text(p.text)[:600])
    print()

In [0]:
# Display 4 sample chunks from each PDF straight from the Delta table
TABLE = cfg.full_table_name

doc_names = [r["document_name"] for r in
             spark.sql(f"SELECT DISTINCT document_name FROM {TABLE}").collect()]

for name in doc_names:
    safe_name = name.replace("'", "''")
    print(f"\n{'=' * 100}\nSample chunks from: {name}\n{'=' * 100}")
    display(spark.sql(f"""
        SELECT chunk_index, page_number, token_count,
               LEFT(chunk_text, 300) AS chunk_text_preview
        FROM {TABLE}
        WHERE document_name = '{safe_name}'
        ORDER BY chunk_index
        LIMIT 4
    """))

In [0]:
# Verification: document names, page ranges, and both PDFs in ONE Delta table
TABLE = cfg.full_table_name

display(spark.sql(f"""
    SELECT document_id, document_name, document_type,
           COUNT(*)                AS chunk_count,
           COUNT(DISTINCT chunk_id) AS distinct_chunk_ids,
           MIN(page_number)        AS first_page,
           MAX(page_number)        AS last_page
    FROM {TABLE}
    GROUP BY document_id, document_name, document_type
"""))

n_docs = spark.sql(f"SELECT COUNT(DISTINCT document_name) AS n FROM {TABLE}").first()["n"]
n_rows = spark.sql(f"SELECT COUNT(*) AS n FROM {TABLE}").first()["n"]
n_ids = spark.sql(f"SELECT COUNT(DISTINCT chunk_id) AS n FROM {TABLE}").first()["n"]
bad_pages = spark.sql(f"SELECT COUNT(*) AS n FROM {TABLE} WHERE page_number IS NULL OR page_number < 1").first()["n"]

print(f"distinct documents in {TABLE}: {n_docs}")
print(f"total rows: {n_rows} | distinct chunk_ids: {n_ids} | invalid page numbers: {bad_pages}")

assert n_docs == 2, "Expected both PDFs in the table"
assert n_rows == n_ids, "chunk_id must be unique"
assert bad_pages == 0, "Every chunk needs a valid page number"
print("\nVERIFIED: both documents stored in one Delta table with unique chunk IDs and valid page numbers")

In [0]:
# Independent schema check: every required metadata column must exist in the Delta table
required_cols = {"document_id", "document_name", "document_type", "chunk_id",
                 "chunk_text", "page_number", "created_at"}

schema = {r["col_name"] for r in spark.sql(f"DESCRIBE TABLE {cfg.full_table_name}").collect()}
assert required_cols.issubset(schema), f"Missing columns: {required_cols - schema}"

print("All required metadata columns present:", sorted(required_cols))
display(spark.sql(f"DESCRIBE TABLE {cfg.full_table_name}"))

In [0]:
# Idempotency test: run ingestion a second time and confirm no duplicate chunks
before = spark.sql(f"SELECT COUNT(*) AS n FROM {cfg.full_table_name}").first()["n"]

summary_2nd = run_ingestion(cfg, spark)  # second execution

after = spark.sql(f"SELECT COUNT(*) AS n FROM {cfg.full_table_name}").first()["n"]
dupes = spark.sql(
    f"SELECT chunk_id FROM {cfg.full_table_name} GROUP BY chunk_id HAVING COUNT(*) > 1"
).count()

print(f"row count before 2nd run: {before}")
print(f"row count after  2nd run: {after}")
print(f"duplicated chunk_ids:     {dupes}")

assert after == before, "Second run changed the row count!"
assert dupes == 0, "Duplicate chunks found!"
print("\nVERIFIED: second ingestion created zero duplicate chunks (idempotent + incremental)")

## Stage 1 complete

* Both PDFs ingested from `dev_project.rag_docs.source_pdfs` into `dev_project.rag_docs.document_chunks`.
* Chunks carry `document_id`, `document_name`, `document_type`, `chunk_id`, `chunk_text`, `page_number`, `created_at` (plus `chunk_index` and `token_count` for ordering and quality checks).
* Re-running the pipeline is idempotent (deterministic UUIDs + MERGE + stale-chunk pruning).

**Waiting for approval before Stage 2 (embeddings / Vector Search).**